<center><h1>Xie_Anna_HW3</h1></center>
<br>
<br>

Name: Anna Xie
<br>
Github Username: annajxiee
<br>
USC ID: 2277255370

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [10]:
import pandas as pd
import os
import numpy as np

Get the AReM Data Set

In [ ]:
# (a) Get the AReM Data Set
data_root = '../data'
activities = ['bending1', 'bending2', 'cycling', 'lying', 'sitting', 'standing', 'walking']
cols = ['time', 'avg_rss12', 'var_rss12', 'avg_rss13', 'var_rss13', 'avg_rss23', 'var_rss23']

# descrition: reads one file and returns it as a cleaned table
# input: data path (like one to each dataset.csv)
# output: a cleaned pd.df
def load_instance(path):
    df = pd.read_csv(path, comment='#', header=None, names=cols,
                     sep=r'[,\s]+', engine='python', usecols=range(7))
    df = df.drop(columns='time')
    assert df.shape[1] == 6, f'{path}: expected 6 columns, got {df.shape[1]}'
    # helps me check if the amt. of columsn isn't 6
    if df.shape[0] != 480:
        print(f'note: {path} has {df.shape[0]} rows instead of 480')
    return df

instances = []   # (activity, file_num, dataframe)
# for every activity that was measured, flip through the folder for every dataset.csv
for activity in activities:
    folder = os.path.join(data_root, activity)
    # build the folder path and .join inserts the right slash to specify root path
    for f in os.listdir(folder):
        # for every filename in folder, 
        # make sure the if skips anything that isn't a CSV
        if f.endswith('.csv'):
            file_num = int(f.replace('dataset', '').replace('.csv', ''))
            # pulls the number out of the filename and the int turns it into a number so we can keep track of which instance the data belongs to
            instances.append((activity, file_num, load_instance(os.path.join(folder, f))))
            # loads the file and stores a tuple of the label, the file number, and the associated data frame 

print(len(instances))

note: ../data\sitting\dataset8.csv has 479 rows instead of 480
88


### (b) Test and Train Data

In [ ]:
# (b) Test and Train Data
training, testing = [], []

# for every activity instance in the dataset, unpack each item into three sections
# make sure the df can be split based on criteria
# dataset1+2.csv from bending1+2 and datasets1+2+3.csv => test
# everything else => train
for activity, file_num, df in instances:
    # bending1+2 should specifically be dataset 1 and 2, but otherwise its dataset1-3
    n_test = 2 if activity in ('bending1', 'bending2') else 3
    (testing if file_num <= n_test else training).append((activity, file_num, df))

print(f'{len(training)} train instances, {len(testing)} test instances')   # 69 and 19

69 train instances, 19 test instances


### (c) Feature Extraction

#### i. Research

Mean and median measure the central value of the data, variance and standard deviation to measure the spread of the data around the center. Minimum, maximum, skewness, and kurtosis to measure the asymmetry, bounds, and peaks of the distribution of the data. Quantiles and IQR and root mean square to measure the variance and check for outliers. Some other features include counts of the local maxima and minima, zero-crossings (the number of times the signal crosses the zero baseline), and mean absolute deviation. 

#### ii. Extraction

In [21]:
# extract the min, max, mean, median, sd, Q1, and Q3 from each instance's 6 cols 

# for each csv file, summarize each col into 7 numbers
# new structure: every row is one csv file, every column is one stat number 

# first get everything out of the dataframe
# description: extracts each column grouped by instance number and creates a dictionary of domain features
# input: dataframe
# output: features dictionary
def extract(df):
  feats = {}
  for i, col in enumerate(df.columns, start=1):
    s = df[col] # pulls the entire col's data
    feats[f'min{i}'] = s.min()
    feats[f'max{i}'] = s.max()
    feats[f'mean{i}'] = s.mean()
    feats[f'median{i}'] = s.median()
    feats[f'std{i}'] = s.std()
    feats[f'Q1-{i}'] = s.quantile(0.25)
    feats[f'Q3-{i}'] = s.quantile(0.75)

  return feats

# description: builds data frame of domain-features relative to the instance (1 out of 88)
# input: instances (the og training/testing df)
# output: a new df w extracted time-domain features per instance 
def build_feats_table(instances):
  rows = []
  for activity, file_num, df in instances:
    row = {'activity': activity, 'file_num': file_num}
    row.update(extract(df))
    rows.append(row)

  return pd.DataFrame(rows)

training_new = build_feats_table(training)
testing_new = build_feats_table(testing)

all_features = pd.concat([training_new, testing_new], ignore_index=True)
all_features.insert(0, 'Instance', range(1, len(all_features) + 1))

print(training_new.shape, testing_new.shape, all_features.shape)   # (69, 44) (19, 44) (88, 44)
all_features.head()
print(all_features.shape)
print(all_features)

(69, 44) (19, 44) (88, 45)
(88, 45)
    Instance  activity  file_num   min1   max1      mean1  median1      std1  \
0          1  bending1         3  35.00  47.40  43.954500   44.330  1.558835   
1          2  bending1         4  33.00  47.75  42.179812   43.500  3.670666   
2          3  bending1         5  33.00  45.75  41.678063   41.750  2.243490   
3          4  bending1         6  37.00  48.00  43.454958   43.250  1.386098   
4          5  bending1         7  36.25  48.00  43.969125   44.500  1.618364   
..       ...       ...       ...    ...    ...        ...      ...       ...   
83        84  standing         2  35.50  46.25  43.174938   43.670  1.989052   
84        85  standing         3  32.75  47.00  42.760562   44.500  3.398919   
85        86   walking         1  19.33  43.50  34.227771   35.500  4.889576   
86        87   walking         2  12.50  45.00  33.509729   34.125  4.850923   
87        88   walking         3  15.00  46.75  34.660583   35.000  5.315110   

   

#### iii. Standard Deviation

In [29]:
# estimate the standard deviation of each of the time-domain features you extracted from the data 
# use python boostrapped to build a 90% CI for sd of each feature 

# isolate the feature names themselves
# from the finished df, get only the feat names so not activity, Instance, or file_num
feat_cols = [c for c in all_features.columns if c not in ('Instance', 'activity', 'file_num')]
# print(feat_cols)

X = all_features[feat_cols].values
# print(type(X))
r, c = X.shape
# r = rows, c = cols

nan_counts = all_features[feat_cols].isna().sum()
print(nan_counts[nan_counts > 0])          # which features contain NaN, and how many

rows = all_features[feat_cols].isna().any(axis=1)
print(all_features.loc[rows, ['Instance', 'activity', 'file_num']])   # which instances 
# point estimate: sample stf of each feat
point_std = X.std(axis=0, ddof=1)

# bootstrap: resample 88 instances w replacement,
# recompute every features's standard deviation,
# repeat a certain number of times 
#spread of sd will show how uncertain the estimate iss 
num = np.random.default_rng(42)
times = 2000
boot_stds = np.empty((times, c))

boot_stds = np.array([
    X[num.integers(0, r, size=r)].std(axis=0, ddof=1)
    for _ in range(times)
])

print(boot_stds.shape, np.isnan(boot_stds).sum())
# print(boot_stds)
print(point_std)

# 90% percentile interval: 5% and 95% 
lower = np.percentile(boot_stds, 5, axis=0)
upper = np.percentile(boot_stds, 95, axis=0)

ci_table = pd.DataFrame({
  'std': point_std,
  'ci_lower': lower,
  'ci_upper': upper
}, index=feat_cols)

ci_table.round(4)
# print(type(point_std), type(lower), type(upper))
# print(np.isnan(boot_stds).sum())    # should be 0
# print(boot_stds.shape)              # should be (2000, 42)
# print(np.isnan(X).sum(), X.shape, X.dtype)              # expect 0, (88, 42), float64
# print((np.isnan(boot_stds).sum(axis=1) > 0).sum())     # how many bootstrap rows contain NaN
# print(np.isnan(boot_stds).sum(axis=0))                 # NaN count per feature


Series([], dtype: int64)
Empty DataFrame
Columns: [Instance, activity, file_num]
Index: []
(2000, 42) 0
[9.56997528 4.39436168 5.33571766 5.44005414 1.7721534  6.15358972
 5.13892461 0.         5.06272937 1.57416392 1.41224416 0.88410548
 0.94638619 2.12526641 2.95646206 4.87513669 4.00838013 4.03639632
 0.94671026 4.22065788 4.17162822 0.         2.18362548 1.16611409
 1.14558566 0.4582417  0.84361968 1.55250418 6.12400143 5.7412383
 5.6755932  5.81378244 1.02489791 6.0964652  5.53172022 0.04583815
 2.51892098 1.15481192 1.08647433 0.51761747 0.75858382 1.52359913]


,std,ci_lower,ci_upper
min1,9.5700,8.1967,10.7872
max1,4.3944,3.3196,5.3201
mean1,5.3357,4.7330,5.8749
median1,5.4401,4.8273,5.9899
std1,1.7722,1.5797,1.9624
Q1-1,6.1536,5.5693,6.6352
Q3-1,5.1389,4.3314,5.8185
min2,0.0000,0.0000,0.0000
max2,5.0627,4.6021,5.4429
mean2,1.5742,1.4008,1.7052


#### iv. Select Features

I believe the three most imoprtant time-domain features are mean, standard deviation amd max. The mean records the average sensor level per feature, which is important for establishing a baseline. The standard deviation calculates the flunctuations and spread of the sensor data and the max shows the pea values of the data. The mins too often show a 0 which might not be helpful for distinguishing activities and these numbers overall capture different, relevant, and diverse information. 

## 2. ISLR 3.7.4

### (a) Linear Train

We can expect the training data RSS of the cubic regression to be lesser than or equal to the linear regression's training RSS. This is because the true relationship is confirmed to be linear. Linear regresion is a more rigid model than cubic regression because cubic has more coefficients, more flexibility so the model has a higher chance of absorbing the random noise and contains a special case where additional coefficients might amount to 0. Training RSS goes down as flexibility increases, so cubic regression, being the more flexible model will score lower.

### (b) Linear Test

The test RSS can expect the linear regression model to have lower RSS. Because the true relationship is linear, the cubic model might be more bent to noise than a straight-line pattern so the unecessary coefficients can include more error, making it overfit. The noise the cubic model's trained to won't repear in test data so the overfit will hurt accuracy during test while maintaining a high bias. Higher variance in this case makes the test RSS for cubic regression greater than the linear. 

### (c) Not Linear Train

We can expect the train RSS for cubis regression to be lower, even more so now that the true relationsip isn't linear and that there's a higher change the cubic regression's additional terms will capture genuine point fluctuation, making it more flexible than linear models. 

### (d) Not Linear Testing

In this case, there isn't enough information to tell how unlinear the data is, whether n=100 is a large enough sample size to even gauge, or how much noise is in the data which are important factors in judging if test RSS for a specific model will be better or not. If the relationship really is strongly nonlinear, then cubic RSS will be lower because the linear model's bias is large and cubic is more flexible. If the relationship is slightly non-linear, the linear model can still be larger on test RSS because the high variance on cubic model would make it too sensitive to noise. 

## 3. ISLR 3.7.3 - Extra Practice 

## 3. ISLR 3.7.5 - Extra Practice 